In [ ]:
### Data Encodeing
 
# 1.  Nominal / OHE Encoding
# 2. Label or Ordinal Encoding 
# 3. Target guided Original Encoding

# One-Hot Encoding (OHE) in Machine Learning

**One-Hot Encoding (OHE)** is a categorical data preprocessing technique used to convert categorical (nominal) variables into a binary vector format suitable for machine learning algorithms.

---

## 1. How It Works

For a categorical column with $N$ unique categories, OHE creates $N$ new binary columns (dummy variables):
* A value of **`1`** (High/Hot) indicates the presence of that category.
* A value of **`0`** (Low/Cold) indicates the absence of that category.

### Example Transformation

| Feature: `Color` |
|---|
| Red |
| Green |
| Blue |

*Transformed via One-Hot Encoding:*

| Color_Red | Color_Green | Color_Blue |
|:---:|:---:|:---:|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

---

## 2. Why It Is Used

* **Prevents Artificial Ordinal Bias:** Algorithms like Linear Regression, SVMs, and Neural Networks treat numerical values as continuous. Label Encoding (`Red=0, Green=1, Blue=2`) causes model algorithms to incorrectly assume an ordering ($\text{Blue} > \text{Green} > \text{Red}$) or distance relationship. OHE ensures all categories are treated equally.
* **Compatibility:** Most mathematical ML algorithms require numeric matrices as input.

---

## 3. Key Considerations

* **Dummy Variable Trap (Multicollinearity):** 
  * Since the binary columns are collinear ($\sum \text{columns} = 1$), you can predict one column from the rest.
  * *Solution:* Drop one column ($N-1$ columns total) when using linear/logistic models (`drop='first'`).
* **High Cardinality Issue:** 
  * If a feature has hundreds of unique values, OHE creates hundreds of sparse columns, leading to high memory consumption and the **Curiosity of Dimensionality**.
  * *Solution:* Use target encoding, frequency encoding, or feature embeddings for high-cardinality data.



In [1]:
import pandas as pd 
from sklearn.preprocessing import OneHotEncoder

df= pd.DataFrame(
    {
        "color" : ["red" , "blue" , 'green', "green", "red" ,"blue"]
    }
)

In [2]:
df

,color
0,red
1,blue
2,green
3,green
4,red
5,blue


In [15]:
# create an instance of onehotencoder

encoder = OneHotEncoder()

In [18]:
# perform fit and then transform 
encoded =encoder.fit_transform(df[['color']]).toarray()
# here rememeber the sorting it will be in alphabetical order 


In [19]:
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out())

In [20]:
encoded_df

,color_blue,color_green,color_red
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,0.0,1.0,0.0
4,0.0,0.0,1.0
5,1.0,0.0,0.0


In [22]:
#for new data

encoder.transform([['blue']]).toarray()

c:\Users\kedar\OneDrive\Documents\Accenture\Kedar\AI_Engineering\MachineLearning\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


array([[1., 0., 0.]])

In [26]:
pd.concat([df,encoded_df] , axis=1)

,color,color_blue,color_green,color_red
0,red,0.0,0.0,1.0
1,blue,1.0,0.0,0.0
2,green,0.0,1.0,0.0
3,green,0.0,1.0,0.0
4,red,0.0,0.0,1.0
5,blue,1.0,0.0,0.0


# One-Hot Encoding: Disadvantages, Sparse Matrices, and Overfitting

While **One-Hot Encoding (OHE)** is essential for converting categorical data into binary vectors, it introduces several structural and statistical challenges in machine learning workflows.

---

## 1. Primary Disadvantages of One-Hot Encoding

* **Dimensionality Explosion (Curse of Dimensionality):**
  Encoding a categorical feature with $N$ unique values adds $N$ (or $N-1$) new columns to the dataset. For high-cardinality features (e.g., ZIP codes, user IDs, device models), this drastically expands feature space.

* **High Memory Consumption:**
  Storing hundreds or thousands of newly generated binary columns increases RAM usage during model training and preprocessing.

* **Information Loss in Tree-Based Models:**
  Decision trees and Random Forests perform poorly on high-cardinality OHE features because splitting on a single sparse binary column (`0` vs. `1`) yields a very small information gain compared to a dense numeric split.

---

## 2. Sparse Matrices

When One-Hot Encoding is applied to high-cardinality features, the resulting dataset consists almost entirely of zeros. This creates a **Sparse Matrix**.

### What Is a Sparse Matrix?
A matrix is considered sparse when the vast majority of its elements have a value of `0`. In OHE, every encoded row contains only a single `1` across all generated dummy columns for a given feature, while every other column holds `0`.

### Impact on Machine Learning Systems

* **Computation Inefficiency:**
  Standard matrix operations (multiplication, inversion) perform unnecessary mathematical operations on zeros ($x \times 0 = 0$), wasting CPU/GPU clock cycles.

* **Storage Overhead:**
  A dense array representation stores every `0` explicitly in memory.

* **Mitigation (Sparse Data Structures):**
  Libraries like `scikit-learn` address this by outputting sparse matrix formats (e.g., **CSR - Compressed Sparse Row**), which only store non-zero values (`1`s) alongside their row and column coordinates.

---

## 3. Overfitting Risk

One-Hot Encoding significantly increases the risk of **overfitting**, particularly when dealing with small datasets or high-cardinality features.

### How OHE Drives Overfitting

1. **Increased Model Capacity (High Variance):**
  Adding many binary columns increases model parameters (e.g., weights in linear models or neural networks). A high ratio of features relative to sample size ($p \gg n$) allows the model to memorize noise instead of learning generalizable patterns.

2. **Spurious Correlations:**
  Categories with very low frequency (e.g., a device type that appears only 2 times in 10,000 rows) create binary columns with almost all `0`s. The model may assign large weights to these isolated `1`s, memorizing target outcomes for rare inputs.

3. **Multicollinearity (Dummy Variable Trap):**
  In linear models, including all $N$ OHE columns creates perfect multicollinearity ($\sum \text{columns} = 1$). This makes model weights unstable and highly sensitive to small training data variations unless $1$ column is dropped (`drop='first'`) or regularization is applied.

---

## 4. Summary & Alternatives

| Issue | Cause | Solution / Alternative |
|---|---|---|
| **Sparse Matrix / Memory Overhead** | Storing millions of redundant zeros | Use CSR/CSC sparse matrices or target encoding |
| **Overfitting** | High feature-to-sample ratio ($p \gg n$) | Apply L1/L2 regularization or group rare categories |
| **Tree Model Inefficiency** | Binary splits yield minimal information gain | Use **Target Encoding**, **Frequency Encoding**, or native categorical support (e.g., CatBoost, LightGBM) |